# S7: Empacotamento congelado do modelo

Este notebook é uma camada fina de publicação. As tabelas são carregadas do cache S7 e do manifesto quando o full existe. A entrada do estimator é exclusivamente `en-US`; a validation foi reutilizada apenas para a calibração final e não é evidência independente. Test, stress e monitor permanecem selados.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'dataset' / 'processed' / 'complaints.parquet').exists():
    parent = PROJECT_ROOT.parent
    if parent == PROJECT_ROOT:
        raise FileNotFoundError('Could not find project root')
    PROJECT_ROOT = parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from consumer_complaint_intelligence.s7 import run_s7, run_s7_smoke
from consumer_complaint_intelligence.s7_reporting import load_s7_report_tables

RUN_MODE = 'disabled'
if RUN_MODE not in {'disabled', 'smoke', 'full'}:
    raise ValueError('RUN_MODE must be disabled, smoke, or full')

scientific_cache = PROJECT_ROOT / 'temp' / 's3' / 'scientific.parquet'
result_artifact = PROJECT_ROOT / 'temp' / 's7' / 's7_results.json'
smoke_artifact = PROJECT_ROOT / 'temp' / 's7' / 's7_smoke_results.json'
frozen_config = PROJECT_ROOT / 'config' / 's7_frozen_package.json'
public_manifest = PROJECT_ROOT / 'config' / 's7_results.json'
bundle = PROJECT_ROOT / 'artifacts' / 's7' / 'consumer_complaint_classifier_s7.joblib'

if RUN_MODE == 'full':
    run_s7(scientific_cache, result_artifact, frozen_config,
           bundle_path=bundle, manifest_path=public_manifest)
elif RUN_MODE == 'smoke':
    run_s7_smoke(scientific_cache, smoke_artifact, frozen_config,
                 bundle_path=bundle)

evidence = None
if public_manifest.exists() and result_artifact.exists() and bundle.exists():
    evidence = load_s7_report_tables(
        result_artifact, manifest_path=public_manifest,
        bundle_path=bundle, config_path=frozen_config)
elif smoke_artifact.exists():
    evidence = load_s7_report_tables(smoke_artifact)

if evidence is not None:
    display(evidence.statuses)
    display(evidence.calibration_summary)
    display(evidence.per_class)
else:
    print('S7 report is cache-only and remains disabled by default.')


status,run_mode,development_only,deploy,confirmatory,input_language,validation_role,validation_independence
str,str,bool,bool,bool,str,str,str
"""packaged_for_confirmation""","""full""",true,false,false,"""en-US""","""FINAL_CALIBRATION_ONLY""","""NOT_INDEPENDENT_EVIDENCE_AFTER…"


status,candidate,threshold,macro_f1,critical_f1,critical_precision,critical_recall,gates_passed,runtime_seconds
str,str,f64,f64,f64,f64,f64,bool,f64
"""packaged_for_confirmation""","""linear_svc_c_0_3_balanced""",0.113535,0.721989,0.292994,0.397313,0.232063,true,267.289378


product_family,precision,recall,f1,support
str,f64,f64,f64,i64
"""cards_prepaid""",0.732497,0.777341,0.754253,17282
"""consumer_lending""",0.558474,0.730112,0.632862,6436
"""credit_reporting""",0.951616,0.911554,0.931154,164801
"""debt_collection""",0.672029,0.722237,0.696229,25565
"""debt_credit_management""",0.397313,0.232063,0.292994,892
"""deposit_accounts""",0.788276,0.826543,0.806956,15635
"""money_services""",0.650883,0.707434,0.677981,5421
"""mortgage""",0.81285,0.917004,0.861791,6181
"""student_loan""",0.808666,0.881869,0.843683,3767
